# insurance-frequency-severity

**Sarmanov copula joint frequency-severity modelling for UK personal lines.**

This notebook challenges the independence assumption that underlies every
standard two-part GLM pricing model: `Pure premium = E[N|x] × E[S|x]`.

That multiplication assumes claim count and average severity are independent
given rating factors. In UK motor, NCD suppresses borderline claims — policyholders
near the NCD threshold do not report near-misses. The result is a systematic
negative correlation between frequency and severity.

This library measures that correlation and corrects the premium for it.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burning-cost/insurance-frequency-severity/blob/main/notebooks/quickstart.ipynb)

In [ ]:
!pip install -q insurance-frequency-severity statsmodels numpy pandas

## Synthetic motor book

We generate 10,000 motor policies with a negative frequency-severity correlation
(omega < 0). This mimics the NCD suppression effect: policyholders who claim
frequently tend to have smaller individual claims (they report everything);
low-frequency policyholders report only large events.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

rng = np.random.default_rng(42)
n = 10_000

# Rating factors
age    = rng.integers(18, 70, n).astype(float)
ncd    = rng.integers(0, 9, n).astype(float)
area   = rng.integers(0, 5, n).astype(float)

# True frequency and severity parameters
log_mu_n = -2.5 + 0.3 * (age < 25) - 0.08 * ncd + 0.1 * area
log_mu_s = 6.5  - 0.03 * ncd  # severity slightly lower for high NCD

mu_n = np.exp(log_mu_n)   # claim frequency (mean ~0.08)
mu_s = np.exp(log_mu_s)   # mean severity (~£600)

# Simulate with negative correlation: high-frequency policies get smaller claims
# True omega (Sarmanov dependence) = -0.25
claim_count = rng.poisson(mu_n)
# Severity: reduce for policies with >1 claims (NCD suppression effect)
sev_scale = mu_s / 3.0 * np.exp(-0.2 * claim_count)  # negative dependence
avg_severity = np.where(
    claim_count > 0,
    rng.gamma(shape=3.0, scale=sev_scale),
    np.nan
)

claims_df = pd.DataFrame({
    "age": age, "ncd": ncd, "area": area,
    "claim_count": claim_count,
    "avg_severity": avg_severity,
})

print(f"Policies: {n:,}")
print(f"Claims:   {(claim_count > 0).sum():,} with at least one claim ({(claim_count > 0).mean():.1%})")
print(f"Mean severity (when claimed): £{np.nanmean(avg_severity):.0f}")
claims_df.head()

## Step 1: Test for dependence

`DependenceTest` runs three tests for frequency-severity dependence:
Spearman rank correlation, a chi-square test on sorted deciles,
and a concordance test on the copula pseudo-observations.

If all three are non-significant, the independence assumption holds and
the standard two-part GLM is fine. If they are significant — proceed.

In [ ]:
from insurance_frequency_severity import DependenceTest

dt = DependenceTest()
test_result = dt.test(claims_df, n_col="claim_count", s_col="avg_severity")

print(f"Spearman rho: {test_result.spearman_rho:.3f}  p={test_result.spearman_p:.4f}")
print(f"Decile test:  chi2={test_result.chi2:.2f}  p={test_result.chi2_p:.4f}")
print()
print(f"Conclusion: {test_result.conclusion}")

## Step 2: Fit marginal GLMs

The Sarmanov copula uses IFM estimation (Inference Functions for Margins):
fit the marginal GLMs first, then estimate the copula parameter omega
from the margins. You plug in your existing GLM objects — no refitting.

In [ ]:
X = claims_df[["age", "ncd", "area"]]
X_const = sm.add_constant(X)

# Frequency GLM: Negative Binomial
freq_glm = sm.GLM(
    claims_df["claim_count"],
    X_const,
    family=sm.families.NegativeBinomial(alpha=0.5)
).fit(disp=False)

# Severity GLM: Gamma (fitted on claims-only rows)
mask = claims_df["claim_count"] > 0
sev_glm = sm.GLM(
    claims_df.loc[mask, "avg_severity"],
    X_const[mask],
    family=sm.families.Gamma(link=sm.families.links.Log())
).fit(disp=False)

print(f"Freq GLM deviance:   {freq_glm.deviance:.1f}")
print(f"Sev GLM deviance:    {sev_glm.deviance:.1f}")

## Step 3: Fit Sarmanov copula and compute premium correction

`JointFreqSev` wraps the two GLMs and estimates omega (the Sarmanov
dependence parameter). `premium_correction()` returns a multiplicative
factor — multiply your standard `E[N] × E[S]` by this to get the
copula-corrected pure premium.

In [ ]:
from insurance_frequency_severity import JointFreqSev

joint = JointFreqSev(freq_glm=freq_glm, sev_glm=sev_glm)
joint.fit(claims_df, n_col="claim_count", s_col="avg_severity")

print(f"Omega (Sarmanov): {joint.omega_:.4f}")
print(f"  Negative omega confirms NCD suppression: high freq = low severity")
print()

# Correction factors for first 5 policies
corrections = joint.premium_correction(X_const[:5])

# Compare standard vs corrected premium
freq_pred = freq_glm.predict(X_const[:5])
sev_pred  = sev_glm.predict(X_const[:5])
standard_pp   = freq_pred * sev_pred
corrected_pp  = standard_pp * corrections

comparison = pd.DataFrame({
    "standard_pp":  np.round(standard_pp, 2),
    "correction":   np.round(corrections, 4),
    "corrected_pp": np.round(corrected_pp, 2),
})
print("Premium correction (first 5 policies):")
print(comparison)

## What you should see

- `DependenceTest` should detect significant dependence (p < 0.05 on at least
  two tests). We built it in.
- Omega should be negative (~-0.1 to -0.3). The exact value depends on
  how tightly the IFM fitting converges on this synthetic dataset.
- `premium_correction()` returns values below 1.0 on average — the independence
  assumption overstates the pure premium when frequency and severity are
  negatively correlated.

## Next steps

- **`GaussianCopulaMixed`** — standard Gaussian copula with PIT approximation
  for discrete margins, for presenting rho in familiar terms
- **`ConditionalFreqSev`** — simplest approach: add N as a covariate in severity GLM
- **`insurance_frequency_severity.dependent`** — neural two-part model with shared
  encoder trunk for large books

**GitHub:** https://github.com/burning-cost/insurance-frequency-severity  
**PyPI:** https://pypi.org/project/insurance-frequency-severity/